In [1]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, DateType, TimestampType
from pyspark.sql.functions import col



import pyspark.sql.functions as F
from pyspark.sql import Window
import json

In [2]:
sc_conf = SparkConf()\
    .set("spark.executor.memory", "4g")\
    .set("spark.executor.cores", "2")\
    .set("spark.hadoop.io.nativeio.enabled", "false")
#add extra parameters if required

In [3]:
spark = SparkSession.builder\
	.appName("Daily Customer Transaction Summary")\
	.config(conf=sc_conf)\
	.getOrCreate()

In [4]:
with open("C:\\Users\\Jabir\\Downloads\\Miscellanious\\Spark\\Bankdata\\BankingDataSparkJob\\configs\\config.json") as f:
     config = json.load(f)

db_conf = config["properties"]

source_path = db_conf["source_path"]
target_path = db_conf["target_path"]    
cdc_key = db_conf["cdc_key"]
cdc_columns = db_conf["cdc_columns"]
watermark = db_conf["watermark"]

print("Source Path: ", source_path)
print("Target Path: ", target_path)
print("CDC Key: ", cdc_key)
print("CDC Columns: ", cdc_columns)
print("Watermark: ", watermark)

Source Path:  C:\Users\Jabir\Downloads\Miscellanious\Spark\Bankdata\BankingDataSparkJob\data\source
Target Path:  C:\Users\Jabir\Downloads\Miscellanious\Spark\Bankdata\BankingDataSparkJob\data\target
CDC Key:  customer_id
CDC Columns:  ['customer_id', 'account_id', 'business_date']
Watermark:  2025-02-10T00:00:00


In [5]:
#Define schema for the source data
customer_schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("customer_number", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("segment", StringType(), True),
    StructField("risk_rating", StringType(), True),
    StructField("kyc_flag", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("customer_since", DateType(), True),
    StructField("status_flag", StringType(), True),
    StructField("fatca_flag", StringType(), True),
    StructField("pep_flag", StringType(), True),
    StructField("preferred_language", StringType(), True),
    StructField("marketing_opt_in", StringType(), True),
    StructField("last_updated_ts", TimestampType(), True),
    StructField("source_system", StringType(), True)
])

account_schema = StructType([
    StructField("account_id", LongType(), True),
    StructField("account_number", StringType(), True),
    StructField("customer_id", LongType(), True),
    StructField("account_type", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("open_date", DateType(), True),
    StructField("close_date", DateType(), True),
    StructField("branch_id", LongType(), True),
    StructField("status_flag", StringType(), True),
    StructField("credit_limit", DoubleType(), True),
    StructField("current_balance", DoubleType(), True),
    StructField("available_balance", DoubleType(), True),
    StructField("overdraft_flag", StringType(), True),
    StructField("interest_rate", DoubleType(), True),
    StructField("product_code", StringType(), True),
    StructField("iban", StringType(), True),
    StructField("swift_code", StringType(), True),
    StructField("last_updated_ts", TimestampType(), True),
    StructField("source_system", StringType(), True),
    StructField("account_tier", StringType(), True)
])

transaction_schema = StructType([
    StructField("txn_id", LongType(), True),
    StructField("account_id", LongType(), True),
    StructField("txn_ts", TimestampType(), True),
    StructField("posting_date", DateType(), True),
    StructField("txn_amount", DoubleType(), True),
    StructField("txn_currency", StringType(), True),
    StructField("txn_type_code", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("merchant_cat_code", StringType(), True),
    StructField("auth_code", StringType(), True),
    StructField("reversal_flag", StringType(), True),
    StructField("orig_txn_id", LongType(), True),
    StructField("txn_status", StringType(), True),
    StructField("fee_amount", DoubleType(), True),
    StructField("exchange_rate", DoubleType(), True),
    StructField("txn_description", StringType(), True),
    StructField("txn_country", StringType(), True),
    StructField("created_ts", TimestampType(), True),
    StructField("last_updated_ts", TimestampType(), True),
    StructField("cdc_operation", StringType(), True)
])

branch_schema = StructType([
    StructField("branch_id", LongType(), True),
    StructField("branch_code", StringType(), True),
    StructField("branch_name", StringType(), True),
    StructField("region", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("address_line1", StringType(), True),
    StructField("address_line2", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("manager_id", StringType(), True),
    StructField("open_date", DateType(), True),
    StructField("close_date", DateType(), True),
    StructField("status_flag", StringType(), True),
    StructField("timezone", StringType(), True),
    StructField("last_updated_ts", TimestampType(), True),
    StructField("source_system", StringType(), True),
    StructField("branch_type", StringType(), True),
    StructField("atm_available", StringType(), True),
    StructField("priority_flag", StringType(), True)
])

ref_txn_type_schema = StructType([
    StructField("txn_type_code", StringType(), True),
    StructField("txn_type_desc", StringType(), True),
    StructField("debit_credit_flag", StringType(), True),
    StructField("fee_applicable_flag", StringType(), True),
    StructField("reporting_group", StringType(), True),
    StructField("high_value_threshold", DoubleType(), True),
    StructField("include_in_risk_score", StringType(), True),
    StructField("last_updated_ts", TimestampType(), True),
    StructField("source_system", StringType(), True)
])



In [6]:
#read source data

customer_df = spark.read \
    .option("header", "true") \
    .schema(customer_schema) \
    .csv("C:\\Users\\Jabir\\Downloads\\Miscellanious\\Spark\\Bankdata\\BankingDataSparkJob\\data\\source\\CUSTOMER.csv")

account_df = spark.read \
    .option("header", "true") \
    .schema(account_schema) \
    .csv("C:\\Users\\Jabir\\Downloads\\Miscellanious\\Spark\\Bankdata\\BankingDataSparkJob\\data\\source\\ACCOUNT.csv")

transaction_df = spark.read \
    .option("header", "true") \
    .schema(transaction_schema) \
    .csv("C:\\Users\\Jabir\\Downloads\\Miscellanious\\Spark\\Bankdata\\BankingDataSparkJob\\data\\source\\TRANSACTION.csv")

branch_df = spark.read \
    .option("header", "true") \
    .schema(branch_schema) \
    .csv("C:\\Users\\Jabir\\Downloads\\Miscellanious\\Spark\\Bankdata\\BankingDataSparkJob\\data\\source\\BRANCH.csv")

ref_txn_type_df = spark.read \
    .option("header", "true") \
    .schema(ref_txn_type_schema) \
    .csv("C:\\Users\\Jabir\\Downloads\\Miscellanious\\Spark\\Bankdata\\BankingDataSparkJob\\data\\source\\REF_TXN_TYPE.csv")


In [7]:
#display dataframes
customer_df.show(5)

+-----------+---------------+----------+---------+-------------+-------+-----------+--------+-------+-------+--------------------+------------+--------------+-----------+----------+--------+------------------+----------------+-------------------+-------------+
|customer_id|customer_number|first_name|last_name|date_of_birth|segment|risk_rating|kyc_flag|country|   city|               email|       phone|customer_since|status_flag|fatca_flag|pep_flag|preferred_language|marketing_opt_in|    last_updated_ts|source_system|
+-----------+---------------+----------+---------+-------------+-------+-----------+--------+-------+-------+--------------------+------------+--------------+-----------+----------+--------+------------------+----------------+-------------------+-------------+
|       1001|       CUST0001|       Ali|     Khan|   1985-03-10| RETAIL|        LOW|       Y|     AE|  Dubai|ali.khan@example.com|971500000001|    2015-01-01|     ACTIVE|         N|       N|                EN|        

In [8]:
account_df.show(5)

+----------+--------------+-----------+------------+--------+----------+----------+---------+-----------+------------+---------------+-----------------+--------------+-------------+------------+------------------+----------+-------------------+-------------+------------+
|account_id|account_number|customer_id|account_type|currency| open_date|close_date|branch_id|status_flag|credit_limit|current_balance|available_balance|overdraft_flag|interest_rate|product_code|              iban|swift_code|    last_updated_ts|source_system|account_tier|
+----------+--------------+-----------+------------+--------+----------+----------+---------+-----------+------------+---------------+-----------------+--------------+-------------+------------+------------------+----------+-------------------+-------------+------------+
|      2001|  AE0000000001|       1001| CREDIT_CARD|     AED|2019-01-01|      null|     3001|     ACTIVE|     20000.0|         5000.0|          15000.0|             N|          0.0|   

In [9]:
transaction_df.show(5)

+------+----------+-------------------+------------+----------+------------+-------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+
|txn_id|account_id|             txn_ts|posting_date|txn_amount|txn_currency|txn_type_code|channel|merchant_cat_code|auth_code|reversal_flag|orig_txn_id|txn_status|fee_amount|exchange_rate|     txn_description|txn_country|         created_ts|    last_updated_ts|cdc_operation|
+------+----------+-------------------+------------+----------+------------+-------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+
|  5001|      2001|2025-02-10 08:30:00|  2025-02-10|    -250.0|         AED|     PURCHASE|    POS|             5411|  AUTH123|            N|       null|    POSTED|       0.

In [10]:
branch_df.show(5)

+---------+-----------+---------------+------+-------+-------+---------------+-------------+-----------+------------+----------+----------+----------+-----------+----------+-------------------+-------------+-----------+-------------+-------------+
|branch_id|branch_code|    branch_name|region|country|   city|  address_line1|address_line2|postal_code|       phone|manager_id| open_date|close_date|status_flag|  timezone|    last_updated_ts|source_system|branch_type|atm_available|priority_flag|
+---------+-----------+---------------+------+-------+-------+---------------+-------------+-----------+------------+----------+----------+----------+-----------+----------+-------------------+-------------+-----------+-------------+-------------+
|     3001|      BR001|     Dubai Main|   DXB|     AE|  Dubai|Sheikh Zayed Rd|         null|      00000|971400000001|      M001|2000-01-01|      null|       OPEN|Asia/Dubai|2025-02-09 12:00:00|           HR|     RETAIL|            Y|            N|
|     30

In [11]:
ref_txn_type_df.show(5)

+-------------+--------------------+-----------------+-------------------+---------------+--------------------+---------------------+-------------------+-------------+
|txn_type_code|       txn_type_desc|debit_credit_flag|fee_applicable_flag|reporting_group|high_value_threshold|include_in_risk_score|    last_updated_ts|source_system|
+-------------+--------------------+-----------------+-------------------+---------------+--------------------+---------------------+-------------------+-------------+
|     PURCHASE|Purchase Transaction|                D|                  N|          SPEND|              3000.0|                    Y|2025-02-09 10:00:00|          REF|
|         CASH|     Cash Withdrawal|                D|                  Y|           CASH|              2000.0|                    Y|2025-02-09 10:05:00|          REF|
|       REFUND|  Refund Transaction|                C|                  N|         REFUND|                 0.0|                    N|2025-02-09 10:10:00|       

In [12]:
# filter only active and open records
customer_active_df = customer_df.filter(F.col("status_flag") == "ACTIVE")
account_active_df = account_df.filter(F.col("status_flag") == "ACTIVE")
branch_active_df = branch_df.filter(F.col("status_flag") == "OPEN")


In [13]:
# Filter by CDC operation and watermark
txn_cdc_df = transaction_df.filter(
    (F.col("cdc_operation").isin("INSERT", "UPDATE")) &
    (F.col("last_updated_ts") > F.to_timestamp(F.lit(watermark)))
)


In [14]:
txn_cdc_df.show()

+------+----------+-------------------+------------+----------+------------+-------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+
|txn_id|account_id|             txn_ts|posting_date|txn_amount|txn_currency|txn_type_code|channel|merchant_cat_code|auth_code|reversal_flag|orig_txn_id|txn_status|fee_amount|exchange_rate|     txn_description|txn_country|         created_ts|    last_updated_ts|cdc_operation|
+------+----------+-------------------+------------+----------+------------+-------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+
|  5001|      2001|2025-02-10 08:30:00|  2025-02-10|    -250.0|         AED|     PURCHASE|    POS|             5411|  AUTH123|            N|       null|    POSTED|       0.

In [15]:
# De-duplicate by txn_id using latest last_updated_ts
w_txn = Window.partitionBy("txn_id").orderBy(F.col("last_updated_ts").desc())

txn_dedup_df = txn_cdc_df.withColumn(
    "rn", F.row_number().over(w_txn)
).filter(F.col("rn") == 1).drop("rn")
txn_dedup_df.show()

+------+----------+-------------------+------------+----------+------------+-------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+
|txn_id|account_id|             txn_ts|posting_date|txn_amount|txn_currency|txn_type_code|channel|merchant_cat_code|auth_code|reversal_flag|orig_txn_id|txn_status|fee_amount|exchange_rate|     txn_description|txn_country|         created_ts|    last_updated_ts|cdc_operation|
+------+----------+-------------------+------------+----------+------------+-------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+
|  5001|      2001|2025-02-10 08:30:00|  2025-02-10|    -250.0|         AED|     PURCHASE|    POS|             5411|  AUTH123|            N|       null|    POSTED|       0.

In [16]:
#Exclude reversals and non-posted
txn_clean_df = txn_dedup_df.filter(
    (F.col("reversal_flag") == "N") &
    (F.col("txn_status") == "POSTED")
)


In [17]:

# Join with reference table for debit/credit and thresholds

txn_enriched_df = txn_clean_df.join(
    ref_txn_type_df,
    on="txn_type_code",
    how="left"
)

txn_enriched_df.show()




+-------------+------+----------+-------------------+------------+----------+------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+--------------------+-----------------+-------------------+---------------+--------------------+---------------------+-------------------+-------------+
|txn_type_code|txn_id|account_id|             txn_ts|posting_date|txn_amount|txn_currency|channel|merchant_cat_code|auth_code|reversal_flag|orig_txn_id|txn_status|fee_amount|exchange_rate|     txn_description|txn_country|         created_ts|    last_updated_ts|cdc_operation|       txn_type_desc|debit_credit_flag|fee_applicable_flag|reporting_group|high_value_threshold|include_in_risk_score|    last_updated_ts|source_system|
+-------------+------+----------+-------------------+------------+----------+------------+-------+-----------------+---------+-------------+----

In [18]:
#  Join with ACCOUNT, CUSTOMER, BRANCH


txn_acc_df = txn_enriched_df.join(
    account_active_df,
    on="account_id",
    how="inner"
)

txn_acc_cust_df = txn_acc_df.join(
    customer_active_df,
    on="customer_id",
    how="inner"
)

txn_full_df = txn_acc_cust_df.join(
    branch_active_df.select("branch_id", "region"),
    on="branch_id",
    how="left"
)


In [19]:
# Derive fields and aggregate to target grain
#    Grain: customer_id, account_id, posting_date


# Signed amount: debits negative, credits positive
txn_with_sign_df = txn_full_df.withColumn(
    "signed_amount",
    F.when(F.col("debit_credit_flag") == "D", -F.col("txn_amount"))
     .when(F.col("debit_credit_flag") == "C", F.col("txn_amount"))
     .otherwise(F.lit(0.0))
)

txn_with_sign_df.show()


+---------+-----------+----------+-------------+------+-------------------+------------+----------+------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+--------------------+-----------------+-------------------+---------------+--------------------+---------------------+-------------------+-------------+--------------+------------+--------+----------+----------+-----------+------------+---------------+-----------------+--------------+-------------+------------+------------------+----------+-------------------+-------------+------------+---------------+----------+---------+-------------+-------+-----------+--------+-------+-------+--------------------+------------+--------------+-----------+----------+--------+------------------+----------------+-------------------+-------------+------+-------------+
|branch_id|customer_id|account_id|txn_type_co

In [20]:
# High value indicator at transaction level
txn_with_hv_df = txn_with_sign_df.withColumn(
    "is_high_value_txn",
    F.when(F.abs(F.col("txn_amount")) > F.col("high_value_threshold"), F.lit(1)).otherwise(F.lit(0))
)
txn_with_hv_df.show()

+---------+-----------+----------+-------------+------+-------------------+------------+----------+------------+-------+-----------------+---------+-------------+-----------+----------+----------+-------------+--------------------+-----------+-------------------+-------------------+-------------+--------------------+-----------------+-------------------+---------------+--------------------+---------------------+-------------------+-------------+--------------+------------+--------+----------+----------+-----------+------------+---------------+-----------------+--------------+-------------+------------+------------------+----------+-------------------+-------------+------------+---------------+----------+---------+-------------+-------+-----------+--------+-------+-------+--------------------+------------+--------------+-----------+----------+--------+------------------+----------------+-------------------+-------------+------+-------------+-----------------+
|branch_id|customer_id|acco

In [21]:
txn_with_hv_df.rdd.glom().map(len).collect() 


[3]

In [27]:
txn_with_hv_df.rdd.getNumPartitions()

1

In [22]:
# Aggregation
agg_df = txn_with_hv_df.groupBy(
    "customer_id",
    "customer_number",
    "first_name",
    "last_name",
    "account_id",
    "account_number",
    "posting_date",
    "risk_rating",
    "kyc_flag",
    "country",
    "branch_id",
    "region"
).agg(
    F.sum(
        F.when(F.col("debit_credit_flag") == "D", F.abs(F.col("txn_amount"))).otherwise(F.lit(0.0))
    ).alias("total_debit_amount"),
    F.sum(
        F.when(F.col("debit_credit_flag") == "C", F.col("txn_amount")).otherwise(F.lit(0.0))
    ).alias("total_credit_amount"),
    F.countDistinct("txn_id").alias("txn_count"),
    F.sum(F.when(F.col("channel") == "ATM", 1).otherwise(0)).alias("atm_txn_count"),
    F.sum(F.when(F.col("channel") == "POS", 1).otherwise(0)).alias("pos_txn_count"),
    F.sum(F.when(F.col("channel").isin("ECOM", "ONLINE"), 1).otherwise(0)).alias("online_txn_count"),
    F.max("txn_ts").alias("last_txn_ts"),
    F.sum("signed_amount").alias("net_txn_amount"),
    F.max("is_high_value_txn").alias("any_high_value_txn")
)

agg_df.show()

+-----------+---------------+----------+---------+----------+--------------+------------+-----------+--------+-------+---------+------+------------------+-------------------+---------+-------------+-------------+----------------+-------------------+--------------+------------------+
|customer_id|customer_number|first_name|last_name|account_id|account_number|posting_date|risk_rating|kyc_flag|country|branch_id|region|total_debit_amount|total_credit_amount|txn_count|atm_txn_count|pos_txn_count|online_txn_count|        last_txn_ts|net_txn_amount|any_high_value_txn|
+-----------+---------------+----------+---------+----------+--------------+------------+-----------+--------+-------+---------+------+------------------+-------------------+---------+-------------+-------------+----------------+-------------------+--------------+------------------+
|       1002|       CUST0002|      Sara|    Ahmed|      2002|  AE0000000002|  2025-02-10|     MEDIUM|       Y|     AE|     3002|   SHJ|            5

In [23]:
# Derive final flags and columns
target_df = agg_df.select(
    F.col("customer_id"),
    F.col("customer_number"),
    F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("customer_name"),
    F.col("account_id"),
    F.col("account_number"),
    F.col("posting_date").alias("business_date"),
    F.round(F.col("total_debit_amount"), 2).alias("total_debit_amount"),
    F.round(F.col("total_credit_amount"), 2).alias("total_credit_amount"),
    F.col("txn_count"),
    F.col("atm_txn_count"),
    F.col("pos_txn_count"),
    F.col("online_txn_count"),
    F.when(F.col("any_high_value_txn") > 0, "Y").otherwise("N").alias("high_value_txn_flag"),
    F.col("risk_rating").alias("risk_segment"),
    F.col("kyc_flag"),
    F.col("country"),
    F.col("branch_id"),
    F.col("region").alias("branch_region"),
    F.col("last_txn_ts"),
    F.round(F.col("net_txn_amount"), 2).alias("net_txn_amount")
)

target_df.show()

+-----------+---------------+-------------+----------+--------------+-------------+------------------+-------------------+---------+-------------+-------------+----------------+-------------------+------------+--------+-------+---------+-------------+-------------------+--------------+
|customer_id|customer_number|customer_name|account_id|account_number|business_date|total_debit_amount|total_credit_amount|txn_count|atm_txn_count|pos_txn_count|online_txn_count|high_value_txn_flag|risk_segment|kyc_flag|country|branch_id|branch_region|        last_txn_ts|net_txn_amount|
+-----------+---------------+-------------+----------+--------------+-------------+------------------+-------------------+---------+-------------+-------------+----------------+-------------------+------------+--------+-------+---------+-------------+-------------------+--------------+
|       1002|       CUST0002|   Sara Ahmed|      2002|  AE0000000002|   2025-02-10|            5000.0|                0.0|        1|       

In [24]:
# Drop duplicates at target level (safety)


target_final_df = target_df.dropDuplicates([
    "customer_id", "account_id", "business_date"
])


In [25]:
target_final_df.show()

+-----------+---------------+-------------+----------+--------------+-------------+------------------+-------------------+---------+-------------+-------------+----------------+-------------------+------------+--------+-------+---------+-------------+-------------------+--------------+
|customer_id|customer_number|customer_name|account_id|account_number|business_date|total_debit_amount|total_credit_amount|txn_count|atm_txn_count|pos_txn_count|online_txn_count|high_value_txn_flag|risk_segment|kyc_flag|country|branch_id|branch_region|        last_txn_ts|net_txn_amount|
+-----------+---------------+-------------+----------+--------------+-------------+------------------+-------------------+---------+-------------+-------------+----------------+-------------------+------------+--------+-------+---------+-------------+-------------------+--------------+
|       1001|       CUST0001|     Ali Khan|      2001|  AE0000000001|   2025-02-10|            1250.0|                0.0|        2|       

In [ ]:
# Write target CSV


target_final_df.coalesce(1).write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(f"{target_path}")
